In [ ]:
import os, random, time, shutil, csv
from collections import defaultdict, Counter

import torch
from torch import nn, optim, einsum
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from torchvision import datasets
from PIL import Image
from tqdm import tqdm

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, roc_curve, auc
)
from itertools import cycle

# ------------------------------
# 1. Config
# ------------------------------
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 1
TARGET_PER_CLASS = 2000
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Cost assumption (set for your machine/GPU to show non-zero training cost)
COST_PER_GPU_HOUR = 0.0  # e.g., 0.35 for T4

# Where to save plots / csv
FIG_DIR = "figures_swin"
OUT_DIR = "outputs"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------
# 2. Dataset Class
# ------------------------------
class LungDataset(Dataset):
    def __init__(self, root_dir, transform=None, selected_indices=None):
        self.base = datasets.ImageFolder(root=root_dir)
        self.samples = self.base.samples
        self.classes = self.base.classes
        self.class_to_idx = self.base.class_to_idx
        self.transform = transform
        if selected_indices:
            self.samples = [self.samples[i] for i in selected_indices]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        # return path so we can write CSVs later
        return image, label, path

# ------------------------------
# 3. Transforms
# ------------------------------
train_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.3, hue=0.02),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ------------------------------
# 4. Paths
# ------------------------------
DATA_ROOT = "DataSet"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR  = os.path.join(DATA_ROOT, "valid")
TEST_DIR = os.path.join(DATA_ROOT, "test")

# ------------------------------
# 5. Base Datasets
# ------------------------------
train_base = LungDataset(TRAIN_DIR)
val_base   = LungDataset(VAL_DIR)
test_base  = LungDataset(TEST_DIR)

NUM_CLASSES = len(train_base.classes)
print(f"✅ Found {NUM_CLASSES} classes: {train_base.classes}")

# ------------------------------
# 6. Data Augmentation + Save New Images
# ------------------------------
class_to_idxs = defaultdict(list)
for i, (path, lbl) in enumerate(train_base.samples):
    class_to_idxs[lbl].append((i, path))

AUGMENT_DIR = os.path.join(DATA_ROOT, "augmented")
os.makedirs(AUGMENT_DIR, exist_ok=True)

image_id = 0
aug_train_idx = []

# PIL-only (no ToTensor) so we can .save()
augmentations = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.3),
])

# Save augmented images only for classes with < TARGET_PER_CLASS
for lbl, idx_paths in class_to_idxs.items():
    cls_name = train_base.classes[lbl]
    save_dir = os.path.join(AUGMENT_DIR, cls_name)
    os.makedirs(save_dir, exist_ok=True)

    original_count = len(idx_paths)
    required = max(0, TARGET_PER_CLASS - original_count)
    sampled = random.choices(idx_paths, k=required) if required > 0 else []

    for _, path in sampled:
        img = Image.open(path).convert("RGB")
        img_aug = augmentations(img)   # remains PIL
        save_path = os.path.join(save_dir, f"aug_{image_id}.jpg")
        img_aug.save(save_path)
        image_id += 1

    for i, _ in idx_paths:
        aug_train_idx.append(i)  # keep originals

# Custom dataset that includes original + augmented
class AugmentedLungDataset(Dataset):
    def __init__(self, original_dataset, aug_dir, transform=None, selected_indices=None):
        self.samples = original_dataset.samples.copy()
        self.transform = transform
        self.class_to_idx = original_dataset.class_to_idx
        self.classes = original_dataset.classes

        for root, _, files in os.walk(aug_dir):
            cls_name = os.path.basename(root)
            if cls_name in self.class_to_idx:
                lbl = self.class_to_idx[cls_name]
                for file in files:
                    if file.lower().endswith((".jpg", ".jpeg", ".png")):
                        self.samples.append((os.path.join(root, file), lbl))

        if selected_indices:
            self.samples = [self.samples[i] for i in selected_indices]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label, path

# Final datasets/loaders
train_ds = AugmentedLungDataset(train_base, AUGMENT_DIR, train_transform)
val_ds   = LungDataset(VAL_DIR,   val_transform)
test_ds  = LungDataset(TEST_DIR,  val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          pin_memory=torch.cuda.is_available())

print(f"📦 Final Train Dataset (with Augmentation): {len(train_ds)} samples")
print(f"📦 Dataset sizes - Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

# Quick balance check after augmentation
label_counts = Counter([lbl for _, lbl in train_ds.samples])
print("Class counts in train (post-aug):", {train_ds.classes[k]: v for k, v in label_counts.items()})

# =============================================================
# 8. Custom Swin Transformer (full shifted-window version)
# =============================================================
from einops import rearrange

# -- helper modules --------------------------------------------------
class CyclicShift(nn.Module):
    def __init__(self, displacement):
        super().__init__()
        self.displacement = displacement
    def forward(self, x):
        return torch.roll(x, shifts=(self.displacement, self.displacement), dims=(1, 2))

class Residual(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn
    def forward(self, x, **kwargs):
        return self.fn(x, **kwargs) + x

class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fn = fn
    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, dim),
        )
    def forward(self, x):
        return self.net(x)

def create_mask(window_size, displacement, upper_lower, left_right):
    mask = torch.zeros(window_size ** 2, window_size ** 2)
    if upper_lower:
        mask[-displacement * window_size:, :-displacement * window_size] = float('-inf')
        mask[:-displacement * window_size, -displacement * window_size:] = float('-inf')
    if left_right:
        mask = rearrange(mask, '(h1 w1) (h2 w2) -> h1 w1 h2 w2', h1=window_size, h2=window_size)
        mask[:, -displacement:, :, :-displacement] = float('-inf')
        mask[:, :-displacement, :, -displacement:] = float('-inf')
        mask = rearrange(mask, 'h1 w1 h2 w2 -> (h1 w1) (h2 w2)')
    return mask

def get_relative_distances(window_size):
    indices = torch.tensor(np.array([[x, y] for x in range(window_size) for y in range(window_size)]))
    distances = indices[None, :, :] - indices[:, None, :]
    return distances

# -- windowed attention --------------------------------------------------
class WindowAttention(nn.Module):
    def __init__(self, dim, heads, head_dim, shifted, window_size, relative_pos_embedding):
        super().__init__()
        inner_dim = head_dim * heads
        self.heads = heads
        self.scale = head_dim ** -0.5
        self.window_size = window_size
        self.relative_pos_embedding = relative_pos_embedding
        self.shifted = shifted

        if shifted:
            disp = window_size // 2
            self.cyclic_shift = CyclicShift(-disp)
            self.cyclic_back_shift = CyclicShift(disp)
            self.upper_lower_mask = nn.Parameter(
                create_mask(window_size, disp, upper_lower=True, left_right=False),
                requires_grad=False)
            self.left_right_mask = nn.Parameter(
                create_mask(window_size, disp, upper_lower=False, left_right=True),
                requires_grad=False)

        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)
        if relative_pos_embedding:
            self.relative_indices = get_relative_distances(window_size) + window_size - 1
            self.pos_embedding = nn.Parameter(torch.randn(2 * window_size - 1, 2 * window_size - 1))
        else:
            self.pos_embedding = nn.Parameter(torch.randn(window_size ** 2, window_size ** 2))
        self.to_out = nn.Linear(inner_dim, dim)

    def forward(self, x):
        if self.shifted:
            x = self.cyclic_shift(x)

        b, H, W, _, h = *x.shape, self.heads
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        nw_h, nw_w = H // self.window_size, W // self.window_size

        q, k, v = map(
            lambda t: rearrange(t, 'b (nh wh) (nw ww) (h d) -> b h (nh nw) (wh ww) d',
                                h=h, wh=self.window_size, ww=self.window_size),
            qkv)

        dots = einsum('b h w i d, b h w j d -> b h w i j', q, k) * self.scale

        if self.relative_pos_embedding:
            dots += self.pos_embedding[
                self.relative_indices[:, :, 0],
                self.relative_indices[:, :, 1]
            ]
        else:
            dots += self.pos_embedding

        if self.shifted:
            dots[:, :, -nw_w:] += self.upper_lower_mask
            dots[:, :, nw_w - 1::nw_w] += self.left_right_mask

        attn = dots.softmax(dim=-1)
        out = einsum('b h w i j, b h w j d -> b h w i d', attn, v)
        out = rearrange(
            out,
            'b h (nh nw) (wh ww) d -> b (nh wh) (nw ww) (h d)',
            nh=nw_h, nw=nw_w, wh=self.window_size, ww=self.window_size
        )
        out = self.to_out(out)

        if self.shifted:
            out = self.cyclic_back_shift(out)
        return out

class SwinBlock(nn.Module):
    def __init__(self, dim, heads, head_dim, mlp_dim, shifted, window_size, relative_pos_embedding):
        super().__init__()
        self.attn = Residual(PreNorm(dim, WindowAttention(
            dim, heads, head_dim, shifted, window_size, relative_pos_embedding)))
        self.mlp = Residual(PreNorm(dim, FeedForward(dim, mlp_dim)))
    def forward(self, x):
        x = self.attn(x)
        x = self.mlp(x)
        return x

class PatchMerging(nn.Module):
    def __init__(self, in_channels, out_channels, downscaling_factor):
        super().__init__()
        self.downscaling_factor = downscaling_factor
        self.unfold = nn.Unfold(kernel_size=downscaling_factor, stride=downscaling_factor, padding=0)
        self.linear = nn.Linear(in_channels * downscaling_factor**2, out_channels)
    def forward(self, x):
        b, c, h, w = x.shape
        df = self.downscaling_factor
        nh, nw = h // df, w // df
        x = self.unfold(x)  # (b, c*df*df, nh*nw)
        x = x.view(b, c * df * df, nh, nw).permute(0, 2, 3, 1)
        return self.linear(x)

class StageModule(nn.Module):
    def __init__(self, in_ch, hid_dim, layers, down_factor, heads, head_dim, window_size, relative_pos_embedding):
        super().__init__()
        assert layers % 2 == 0
        self.merge = PatchMerging(in_ch, hid_dim, down_factor)
        self.blocks = nn.ModuleList()
        for _ in range(layers // 2):
            self.blocks.append(nn.ModuleList([
                SwinBlock(hid_dim, heads, head_dim, hid_dim * 4, False, window_size, relative_pos_embedding),
                SwinBlock(hid_dim, heads, head_dim, hid_dim * 4, True,  window_size, relative_pos_embedding),
            ]))
    def forward(self, x):
        x = self.merge(x)
        for regular, shifted in self.blocks:
            x = regular(x)
            x = shifted(x)
        return x.permute(0, 3, 1, 2)

class SwinTransformer(nn.Module):
    def __init__(self, *,
                 hidden_dim, layers, heads,
                 channels=3, num_classes=NUM_CLASSES,
                 head_dim=32, window_size=7,
                 downscaling_factors=(4, 2, 2, 2),
                 relative_pos_embedding=True):
        super().__init__()
        self.stage1 = StageModule(channels,      hidden_dim,     layers[0],
                                  downscaling_factors[0], heads[0], head_dim,
                                  window_size, relative_pos_embedding)
        self.stage2 = StageModule(hidden_dim,     hidden_dim*2,   layers[1],
                                  downscaling_factors[1], heads[1], head_dim,
                                  window_size, relative_pos_embedding)
        self.stage3 = StageModule(hidden_dim*2,   hidden_dim*4,   layers[2],
                                  downscaling_factors[2], heads[2], head_dim,
                                  window_size, relative_pos_embedding)
        self.stage4 = StageModule(hidden_dim*4,   hidden_dim*8,   layers[3],
                                  downscaling_factors[3], heads[3], head_dim,
                                  window_size, relative_pos_embedding)
        self.norm = nn.LayerNorm(hidden_dim * 8)
        self.head = nn.Linear(hidden_dim * 8, num_classes)
    def forward(self, img):
        x = self.stage1(img)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = x.mean(dim=[2, 3])       # global avg pooling
        return self.head(self.norm(x))

# ---------- instantiate and move to device ----------
model = SwinTransformer(
    hidden_dim=96,
    layers=(2,2,6,2),
    heads=(3,6,12,24),
    channels=3,
    num_classes=NUM_CLASSES,
    head_dim=32,
    window_size=7
).to(DEVICE)

print(f"✅ Swin Transformer ready → parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f} M")

# ------------------------------
# 9. Checkpoint + Optimizer
# ------------------------------
CHECKPOINT_PATH = "swin_checkpoint.pth"
RESUME          = True           # set False to start fresh even if a ckpt exists

criterion  = nn.CrossEntropyLoss()
optimizer  = optim.AdamW(model.parameters(), lr=LR)
scheduler  = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

start_epoch   = 0
best_val_acc  = 0.0

if RESUME and os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optim_state"])
    scheduler.load_state_dict(ckpt["sched_state"])
    best_val_acc = ckpt["best_val_acc"]
    start_epoch  = ckpt["epoch"] + 1
    print(f"✅ Resumed from epoch {ckpt['epoch']} | best_val_acc={best_val_acc:.4f}")
else:
    print("ℹ️  No checkpoint found or RESUME=False — starting fresh.")

# ------------------------------
# helpers
# ------------------------------
def _unpack(batch):
    # supports (imgs, labels) or (imgs, labels, paths)
    if len(batch) == 3: return batch
    imgs, labels = batch
    return imgs, labels, None

# ------------------------------
# 10. Training Loop (with history)
# ------------------------------
history = {
    "epochs": [],
    "train_acc": [],
    "val_acc": [],
    "train_loss": [],
    "val_loss": [],
}
epoch_times = []
train_start_time = time.time()

for epoch in range(start_epoch, NUM_EPOCHS):
    print(f"\n🔄 Epoch {epoch+1}/{NUM_EPOCHS}")
    epoch_t0 = time.time()

    model.train()
    running_loss = correct_preds = total_preds = 0

    for batch in tqdm(train_loader, desc="  • Training"):
        images, labels, _ = _unpack(batch)
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct_preds += (outputs.argmax(1) == labels).sum().item()
        total_preds   += labels.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    train_acc  = correct_preds / total_preds
    print(f"    🟢 Train  | loss={train_loss:.4f}  acc={train_acc:.4f}")

    # ---------- Validation ----------
    model.eval()
    val_loss = val_correct = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="  • Validate", leave=False):
            images, labels, _ = _unpack(batch)
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            val_loss    += loss.item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()

    val_loss /= len(val_loader.dataset)
    val_acc   = val_correct / len(val_loader.dataset)
    print(f"    🔵 Val    | loss={val_loss:.4f}  acc={val_acc:.4f}")

    history["epochs"].append(epoch + 1)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    # ---------- Checkpoint ----------
    improved = val_acc > best_val_acc
    if improved:
        best_val_acc = val_acc
        torch.save(
            {
                "epoch":        epoch,
                "model_state":  model.state_dict(),
                "optim_state":  optimizer.state_dict(),
                "sched_state":  scheduler.state_dict(),
                "best_val_acc": best_val_acc,
            },
            CHECKPOINT_PATH,
        )
        print(f"    💾 Saved new best checkpoint (acc={best_val_acc:.4f})")

    scheduler.step()
    print(f"    🔄 LR stepped -> {scheduler.get_last_lr()[0]:.6f}")

    epoch_time = time.time() - epoch_t0
    epoch_times.append(epoch_time)

train_total_time = time.time() - train_start_time
print("\n🎉 Training complete.")

# Reload best
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE)["model_state"])
model.eval()

# ------------------------------
# 11. Helper: preds / probs
# ------------------------------
def get_preds_probs(loader):
    all_logits, all_probs, all_preds, all_labels = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            images, labels, _ = _unpack(batch)
            images = images.to(DEVICE)
            logits = model(images)
            probs  = torch.softmax(logits, dim=1).cpu().numpy()
            preds  = logits.argmax(1).cpu().numpy()
            all_logits.append(logits.cpu().numpy())
            all_probs.append(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return (
        np.concatenate(all_logits, axis=0),
        np.concatenate(all_probs, axis=0),
        np.array(all_preds),
        np.array(all_labels),
    )

print("🔎 Collecting predictions for Train/Val/Test …")
_, train_probs, train_preds, train_labels = get_preds_probs(train_loader)
_, val_probs,   val_preds,   val_labels   = get_preds_probs(val_loader)
_, test_probs,  test_preds,  test_labels  = get_preds_probs(test_loader)

# Also: save test-set per-image CSV
def infer_to_csv(loader, save_path, class_names):
    header = ["filepath", "true_label", "pred_label"] + [f"prob_{c}" for c in class_names]
    rows = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Inference (CSV)", leave=False):
            images, labels, paths = _unpack(batch)
            images = images.to(DEVICE)
            logits = model(images)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = logits.argmax(1).cpu().numpy()
            for pth, y, yhat, probrow in zip(paths, labels.numpy(), preds, probs):
                rows.append([pth, train_base.classes[y], train_base.classes[yhat]] + list(map(float, probrow)))
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "w", newline="") as f:
        writer = csv.writer(f); writer.writerow(header); writer.writerows(rows)

csv_path = os.path.join(OUT_DIR, "swin_test_inference.csv")
infer_to_csv(test_loader, csv_path, train_base.classes)
print(f"📝 Saved per-image test inference CSV → {csv_path}")

# Accuracies
train_acc_final = (train_preds == train_labels).mean()
val_acc_final   = (val_preds == val_labels).mean()
test_acc_final  = (test_preds == test_labels).mean()

avg_kw = {"average": "macro"} if NUM_CLASSES > 2 else {}
prec_test = precision_score(test_labels, test_preds, **avg_kw, zero_division=0)
rec_test  = recall_score(test_labels, test_preds, **avg_kw, zero_division=0)
f1_test   = f1_score(test_labels, test_preds, **avg_kw, zero_division=0)

print(f"✅ Final Accuracies | Train={train_acc_final:.4f}  Val={val_acc_final:.4f}  Test={test_acc_final:.4f}")

# Specificity & Sensitivity
def specificity_per_class(y_true, y_pred, num_classes):
    specs = []
    for c in range(num_classes):
        tn = np.sum((y_true != c) & (y_pred != c))
        fp = np.sum((y_true != c) & (y_pred == c))
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specs.append(spec)
    return np.array(specs), np.mean(specs)

specs, spec_macro = specificity_per_class(test_labels, test_preds, NUM_CLASSES)
sens_per_class_test = recall_score(test_labels, test_preds, labels=list(range(NUM_CLASSES)),
                                   average=None, zero_division=0)
sens_macro = sens_per_class_test.mean()

# Inference time per sample (approx)
def measure_inference_time(loader, n_batches=20):
    model.eval()
    times = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches: break
            images, _, _ = _unpack(batch)
            images = images.to(DEVICE)
            t0 = time.time()
            _ = model(images)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            times.append(time.time() - t0)
    if not times:
        return 0.0
    avg_per_batch = np.mean(times)
    return avg_per_batch / images.size(0)

avg_infer_time = measure_inference_time(test_loader)

# Training cost estimate
train_hours = train_total_time / 3600.0
training_cost = train_hours * COST_PER_GPU_HOUR

# ------------------------------
# 12. PLOTS — Part 1: Model Performance Evaluation
# ------------------------------
model_name = "Swin"
cls_names = train_base.classes

# (1) Train vs Test Accuracy — per model
plt.figure()
plt.bar([f"{model_name}-Train", f"{model_name}-Test"], [train_acc_final, test_acc_final])
plt.ylabel("Accuracy")
plt.title("Training Accuracy vs Testing Accuracy (per model)")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "1_train_vs_test_acc_per_model.png"), dpi=200)
plt.close()

# (2) Specificity vs Sensitivity — grouped bar (macro)
plt.figure()
x = np.arange(1)
width = 0.35
plt.bar(x - width/2, [spec_macro], width, label="Specificity")
plt.bar(x + width/2, [sens_macro], width, label="Sensitivity (Recall)")
plt.xticks(x, [model_name])
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Specificity vs Sensitivity (Test, macro)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "2_specificity_vs_sensitivity.png"), dpi=200)
plt.close()

# (3) Precision, Recall, F1 — grouped bar (Test macro)
plt.figure()
metrics = ["Precision", "Recall", "F1"]
vals = [prec_test, rec_test, f1_test]
plt.bar(metrics, vals)
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Precision / Recall / F1 (Test, macro)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "3_prf1_per_model.png"), dpi=200)
plt.close()

# (4) Confusion Matrix — Test
cm_test = confusion_matrix(test_labels, test_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_test, display_labels=cls_names)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap='Blues', ax=ax, colorbar=False)
plt.title("Confusion Matrix — Test")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "4_confusion_matrix_test.png"), dpi=200)
plt.close()

# Optional: Confusion Matrix — Train
cm_train = confusion_matrix(train_labels, train_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_train, display_labels=cls_names)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap='Greens', ax=ax, colorbar=False)
plt.title("Confusion Matrix — Train")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "4b_confusion_matrix_train.png"), dpi=200)
plt.close()

# ------------------------------
# 13. PLOTS — Part 2: Proposed Methodology
# ------------------------------
# (5) Training vs Testing Accuracy — Model Comparison (single model now)
plt.figure()
labels_cmp = [model_name]
train_vals = [train_acc_final]
test_vals  = [test_acc_final]
x = np.arange(len(labels_cmp))
width = 0.35
plt.bar(x - width/2, train_vals, width, label="Train Acc")
plt.bar(x + width/2, test_vals,  width, label="Test Acc")
plt.xticks(x, labels_cmp)
plt.ylim(0, 1)
plt.ylabel("Accuracy")
plt.title("Training vs Testing Accuracy — Model Comparison")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "5_train_vs_test_acc_comparison.png"), dpi=200)
plt.close()

# (6) Training Time vs Inference Time — grouped bar
plt.figure()
x = np.arange(1)
plt.bar(x - width/2, [train_total_time], width, label="Training Time (s)")
plt.bar(x + width/2, [avg_infer_time],  width, label="Avg Inference Time / sample (s)")
plt.xticks(x, [model_name])
plt.ylabel("Seconds")
plt.title("Training Time vs Inference Time")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "6_time_train_vs_infer.png"), dpi=200)
plt.close()

# (7) Training Cost per model — bar
plt.figure()
plt.bar([model_name], [training_cost])
plt.ylabel("Cost (USD)")
plt.title("Training Cost per Model")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "7_training_cost_per_model.png"), dpi=200)
plt.close()

# (8) Training & Validation Accuracy vs Epochs — line
plt.figure()
plt.plot(history["epochs"], history["train_acc"], label="Train Acc")
plt.plot(history["epochs"], history["val_acc"],   label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.title("Training & Validation Accuracy vs Epochs")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "8_train_val_acc_vs_epochs.png"), dpi=200)
plt.close()

# Helper to plot ROC (binary & multiclass)
def plot_roc_curves(probs, labels, title, save_path):
    n_classes = probs.shape[1]
    y_true_bin = np.eye(n_classes)[labels]
    fpr = dict(); tpr = dict(); roc_auc = dict()

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    fpr["macro"], tpr["macro"] = all_fpr, mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

    plt.figure()
    plt.plot(fpr["macro"], tpr["macro"], lw=2,
             label=f"{model_name} (macro AUC = {roc_auc['macro']:.3f})")
    for i in range(n_classes):
        plt.plot(fpr[i], tpr[i], lw=1, label=f"Class {cls_names[i]} (AUC = {roc_auc[i]:.3f})")
    plt.plot([0, 1], [0, 1], lw=1, linestyle="--")
    plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()

# (9) ROC Curve — Training
plot_roc_curves(train_probs, train_labels,
                title="ROC Curve — Training",
                save_path=os.path.join(FIG_DIR, "9_roc_training.png"))

# (10) ROC Curve — Testing
plot_roc_curves(test_probs, test_labels,
                title="ROC Curve — Testing",
                save_path=os.path.join(FIG_DIR, "10_roc_testing.png"))

print(f"📊 Saved all figures to: {FIG_DIR}/  | CSVs in: {OUT_DIR}/")

# ------------------------------
# 14. Confusion Matrix + Report (display)
# ------------------------------
cm = confusion_matrix(test_labels, test_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=train_base.classes)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix on Test Set (Display)")
plt.tight_layout()
plt.show()

print("\n🧾 Classification Report (Test):\n",
      classification_report(test_labels, test_preds, target_names=train_base.classes, zero_division=0))

# ------------------------------
# 15. LIME (same flow; just ensure packages installed)
# ------------------------------
from lime import lime_image
from skimage.segmentation import mark_boundaries

raw_test_ds = LungDataset(TEST_DIR, transform=None)
sample_img, sample_label, _ = raw_test_ds[0]
img_np = np.array(sample_img)

def predict_proba(images_np):
    model.eval()
    with torch.no_grad():
        images_resized = torch.nn.functional.interpolate(
            torch.tensor(images_np).permute(0, 3, 1, 2).float(),
            size=(IMAGE_SIZE, IMAGE_SIZE), mode='bilinear', align_corners=False
        ) / 255.0
        images_resized = T.Normalize([0.485, 0.456, 0.406],
                                     [0.229, 0.224, 0.225])(images_resized)
        images_resized = images_resized.to(DEVICE)
        outputs = model(images_resized)
        return outputs.softmax(1).cpu().numpy()

explainer = lime_image.LimeImageExplainer()
explanation = explainer.explain_instance(
    image=img_np,
    classifier_fn=predict_proba,
    top_labels=1,
    hide_color=0,
    num_samples=1000
)

temp, mask = explanation.get_image_and_mask(
    label=explanation.top_labels[0],
    positive_only=False,
    hide_rest=False,
    num_features=10,
    min_weight=0.0
)

plt.figure(figsize=(6, 6))
plt.imshow(mark_boundaries(temp / 255.0, mask))
plt.title(f"LIME Explanation for class: {train_base.classes[sample_label]}")
plt.axis('off')
plt.tight_layout()
plt.show()
